In [1]:
import datetime
import datetime as dt

import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from scipy.optimize import minimize
import xarray as xr

from xcube.core.store import new_data_store
from xcube.core.chunk import chunk_dataset

In [ ]:
from dask.distributed import Client, LocalCluster
import dask

cluster = LocalCluster(
    n_workers=12,              
    threads_per_worker=2,
    memory_limit='2GB', # per worker
)
client = Client(cluster)
client

In [2]:
import os

S3_USER_STORAGE_KEY = os.environ["S3_USER_STORAGE_KEY"]
S3_USER_STORAGE_SECRET = os.environ["S3_USER_STORAGE_SECRET"]
S3_USER_STORAGE_BUCKET = os.environ["S3_USER_STORAGE_BUCKET"]

irr_store = new_data_store("s3",
                       root=S3_USER_STORAGE_BUCKET,
                       storage_options=dict(anon=False,
                                            key=S3_USER_STORAGE_KEY,
                                            secret=S3_USER_STORAGE_SECRET))

In [4]:
irr_store.list_data_ids()

['calibrated.zarr',
 'calibrated_0.zarr',
 'calibrated_1.zarr',
 'calibrated_10.zarr',
 'calibrated_11.zarr',
 'calibrated_12.zarr',
 'calibrated_13.zarr',
 'calibrated_14.zarr',
 'calibrated_15.zarr',
 'calibrated_16.zarr',
 'calibrated_17.zarr',
 'calibrated_18.zarr',
 'calibrated_19.zarr',
 'calibrated_2.zarr',
 'calibrated_20.zarr',
 'calibrated_3.zarr',
 'calibrated_4.zarr',
 'calibrated_5.zarr',
 'calibrated_6.zarr',
 'calibrated_7.zarr',
 'calibrated_8.zarr',
 'calibrated_9.zarr',
 'deleteme.zarr',
 'era5.zarr',
 'era5v2.zarr',
 'era5v3_timeopt.zarr',
 'irrigation_input.zarr',
 'irrigation_input_small_chunks.zarr',
 'soil_moisture_filled.zarr',
 'soil_moisture_zappend.zarr']

In [6]:
ds = irr_store.open_data("irrigation_input_small_chunks.zarr")
ds

<xarray.Dataset> Size: 1TB
Dimensions:      (time: 3561, lat: 4144, lon: 6832)
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Data variables:
    SWI          (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>
    pev          (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>
    tp           (time, lat, lon) float32 403GB dask.array<chunksize=(3561, 50, 50), meta=np.ndarray>

In [7]:
#CALIBRATION
def sm_inversion(sm, et, a, b, z, RF, thr=None): 
    """Evotranspiration and Soil moisture to irrigation"""
    # sm - soil moisture
    # et - evotranspiration
    # a/b - drainage parameter
    # z - soil water capacity
    # f/RF - Adjusting factor
    p_sim = z * (sm[1:] - sm[:-1]) + ((a * sm[1:]**b + a * sm[:-1]**b) / 2.) +  ((RF * sm[1:] * et[1:] + RF * sm[:-1] * et[:-1]) / 2.)

    p_sim[abs(np.diff(sm))<=0.001]=0.0
    p_sim[p_sim < 1.0]=0.0   # 2mm/day for NN=4 -> 0.5

    return np.clip(p_sim, 0, thr)

def calib_sm_inversion(sm,  p_obs, et, NN,   x0=None, bounds=None, options=None, method='TNC'): 


    if x0 is None:
        x0 = np.array([20., 5., 80,1.])

    if bounds is None:
        bounds = ((0,200), (0.01, 50), (1, 800), (0.1,1.4))

    # if options is None:
    #     options = {'ftol': 1e-8, 'maxiter': 4000, 'disp': False}

    if options is None:
        options = {'ftol': 1e-8, 'maxfun': 4000, 'disp': False}

#    if options is None:
#        options = {'ftol': 1e-8, 'maxiter': 3000, 'disp': False}


    result = minimize(cost_fun, x0, args=(sm,  p_obs, et, NN),method=method, bounds=bounds, options=options)

    a, b, z,RF = result.x

    return a, b, z, RF


def cost_fun(x0, sm,  p_obs, et, NN):

    # The following args are 1D time-series
    p_sim = sm_inversion(sm, et, x0[0], x0[1], x0[2],x0[3])
    p_obs = p_obs[:-1]
    p_sim[np.isnan(p_obs)] = np.nan
    p_sim1 = np.add.reduceat(p_sim[np.isfinite(p_sim)], np.arange(0, len(p_sim[np.isfinite(p_sim)]), NN))
    p_obs1 = np.add.reduceat(p_obs[np.isfinite(p_sim)], np.arange(0, len(p_obs[np.isfinite(p_sim)]), NN))
    rmsd = np.nanmean((p_obs1 - p_sim1)**2)**0.5

    return rmsd


The calibration function expects a 1D array of single pixel time-series for soil moisture, evotranspiration and rainfall.

Define irrigation season
- may to september (will be refined later)
- dataset that will be published will contain those months and 0 for others - aggregated at 1 or 2 weeks
- from jan-may and sep-dec per year, we take all days for calibration.
- then inside the range may-sep, we also calibrate where rainfall < 1mm pixel-wise
- then based on the calibrated values, we calculate the IWU for may-sep

In [8]:
def calib_wrapper(sm_ts, p_obs_ts, et_ts, NN):
    if np.all(np.isnan(sm_ts)):
        return np.array([np.nan, np.nan, np.nan, np.nan])

    a, b, z, RF = calib_sm_inversion(sm_ts, p_obs_ts, et_ts, NN)
    return np.array([a, b, z, RF])

In [9]:
tp = ds["tp"]
tp

<xarray.DataArray 'tp' (time: 3561, lat: 4144, lon: 6832)> Size: 403GB
dask.array<open_dataset-tp, shape=(3561, 4144, 6832), dtype=float32, chunksize=(3561, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B ...
Attributes: (12/33)
    GRIB_NV:                                  0
    GRIB_Nx:                                  611
    GRIB_Ny:                                  373
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            fc
    ...                                       ...
    GRIB_units:                               mm
    GRIB_uvRelativeToGrid:                    0
    grid_mapping:                             spatial_ref
    long_name:                                Total precipitation (millimeters)
    standard_name:                            unknown
    units:                                    mm

In [10]:
mask_season = tp["time"].dt.month.isin([5, 6, 7, 8, 9])
masked_tp = tp.where(~(mask_season & (tp < 1.0)))
masked_tp

<xarray.DataArray 'tp' (time: 3561, lat: 4144, lon: 6832)> Size: 403GB
dask.array<where, shape=(3561, 4144, 6832), dtype=float32, chunksize=(3561, 50, 50), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 28kB 2016-01-01 2016-01-02 ... 2025-09-30
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0
Attributes: (12/33)
    GRIB_NV:                                  0
    GRIB_Nx:                                  611
    GRIB_Ny:                                  373
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            fc
    ...                                       ...
    GRIB_units:                               mm
    GRIB_uvRelativeToGrid:                    0
    grid_mapping:                             spatial_ref
    long_name:                                Total precipitation (millimeters)
    standard_name:                            unknown
    units:                                    mm

In [11]:
result = xr.apply_ufunc(
    calib_wrapper,
    ds["SWI"],      
    masked_tp,    
    ds["pev"],       
    7,           
    input_core_dims=[["time"], ["time"], ["time"], []],  
    output_core_dims=[["params"]],  
    vectorize=True,                 
    dask="parallelized",           
    output_dtypes=[float],
    output_sizes={"params": 4}
)
result

/tmp/ipykernel_3047/4231297673.py:1: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  result = xr.apply_ufunc(


<xarray.DataArray (lat: 4144, lon: 6832, params: 4)> Size: 906MB
dask.array<transpose, shape=(4144, 6832, 4), dtype=float64, chunksize=(50, 50, 4), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
    spatial_ref  int64 8B 0
Dimensions without coordinates: params

In [12]:
result = result.assign_coords(params=["a", "b", "z", "RF"])
result

<xarray.DataArray (lat: 4144, lon: 6832, params: 4)> Size: 906MB
dask.array<transpose, shape=(4144, 6832, 4), dtype=float64, chunksize=(50, 50, 4), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * params       (params) <U2 32B 'a' 'b' 'z' 'RF'
    spatial_ref  int64 8B 0

In [13]:
result = result.to_dataset(name="calibration")
result

<xarray.Dataset> Size: 906MB
Dimensions:      (lat: 4144, lon: 6832, params: 4)
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * params       (params) <U2 32B 'a' 'b' 'z' 'RF'
    spatial_ref  int64 8B 0
Data variables:
    calibration  (lat, lon, params) float64 906MB dask.array<chunksize=(50, 50, 4), meta=np.ndarray>

In [14]:
subresults = []
step = 200
for i in range(0, result.sizes["lat"], step):
    subresults.append(result.isel(lat=slice(i, i+step)))

In [15]:
%%time
for i, subresult in enumerate(subresults):
    print(i)
    if irr_store.has_data(f"calibrated_{i}.zarr"):
        continue
    irr_store.write_data(subresult, f"calibrated_{i}.zarr")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20


/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_3047/2242989319.py:50: 

CPU times: user 12min 9s, sys: 2min 20s, total: 14min 30s
Wall time: 2h 19min 48s


In [8]:
%%time
data_ids_cal = [f"calibrated_{i}.zarr" for i in range(21)]
data_ids_cal

CPU times: user 15 μs, sys: 4 μs, total: 19 μs
Wall time: 21 μs


['calibrated_0.zarr',
 'calibrated_1.zarr',
 'calibrated_2.zarr',
 'calibrated_3.zarr',
 'calibrated_4.zarr',
 'calibrated_5.zarr',
 'calibrated_6.zarr',
 'calibrated_7.zarr',
 'calibrated_8.zarr',
 'calibrated_9.zarr',
 'calibrated_10.zarr',
 'calibrated_11.zarr',
 'calibrated_12.zarr',
 'calibrated_13.zarr',
 'calibrated_14.zarr',
 'calibrated_15.zarr',
 'calibrated_16.zarr',
 'calibrated_17.zarr',
 'calibrated_18.zarr',
 'calibrated_19.zarr',
 'calibrated_20.zarr']

In [9]:
%%time
datasets = []
for data_id in data_ids_cal:
    datasets.append(irr_store.open_data(data_id))
ds = xr.concat(datasets, dim="lat", join="left")
ds

CPU times: user 6.88 s, sys: 45.1 ms, total: 6.92 s
Wall time: 17.2 s


<xarray.Dataset> Size: 906MB
Dimensions:      (lat: 4144, lon: 6832, params: 4)
Coordinates:
  * lat          (lat) float64 33kB 72.0 71.99 71.98 71.97 ... 35.02 35.01 35.0
  * lon          (lon) float64 55kB -11.0 -10.99 -10.98 ... 49.98 49.99 50.0
  * params       (params) <U2 32B 'a' 'b' 'z' 'RF'
    spatial_ref  int64 8B 0
Data variables:
    calibration  (lat, lon, params) float64 906MB dask.array<chunksize=(50, 50, 4), meta=np.ndarray>

In [11]:
%%time
irr_store.write_data(ds, "calibrated.zarr", replace=True)

CPU times: user 25.1 s, sys: 1.86 s, total: 27 s
Wall time: 1min 4s


'calibrated.zarr'

In [ ]:
%%time
for data_id in data_ids_cal:
    irr_store.delete_data(data_id)

## Quick validation

In [3]:
calibrated = irr_store.open_data("calibrated.zarr")

In [4]:
arr_reshaped = calibrated.calibration.values.reshape(-1, 4)

In [5]:
unique_param_sets = np.unique(arr_reshaped, axis=0)

print(f"Number of unique parameter sets: {len(unique_param_sets)}")
print(unique_param_sets)

Number of unique parameter sets: 28143508
[[0.         0.01       1.         0.1       ]
 [0.         0.01       1.         0.15857308]
 [0.         0.01       1.         0.28535717]
 ...
 [       nan        nan        nan        nan]
 [       nan        nan        nan        nan]
 [       nan        nan        nan        nan]]


In [6]:
# Should be more than 50k or so (could be quite higher)
unique_param_sets_no_nan = unique_param_sets[~np.isnan(unique_param_sets).any(axis=1)]
print(f"Unique parameter sets without NaNs: {len(unique_param_sets_no_nan)}")
print(unique_param_sets_no_nan)

Unique parameter sets without NaNs: 4927398
[[0.00000000e+00 1.00000000e-02 1.00000000e+00 1.00000000e-01]
 [0.00000000e+00 1.00000000e-02 1.00000000e+00 1.58573083e-01]
 [0.00000000e+00 1.00000000e-02 1.00000000e+00 2.85357175e-01]
 ...
 [2.00000000e+02 4.99048892e+01 2.92917945e+01 1.40000000e+00]
 [2.00000000e+02 4.99999976e+01 4.19882613e+01 1.40000000e+00]
 [2.00000000e+02 5.00000000e+01 1.09321905e+02 1.00000000e-01]]
